# Bayesian A/B Testing Methods — Complete Solution

Fully worked Beta-Binomial analysis, decision metrics, posterior visualisations, alternate implementations, extra practice answers, Monte-Carlo verification and a simulation playground.

---


## Flowchart: Bayesian A/B Testing Decision Process

This flowchart shows the recommended Bayesian workflow.  Unlike classical fixed-horizon tests, Bayesian methods allow continuous monitoring and produce directly interpretable probability statements that are easy to communicate to different audiences.

```mermaid
flowchart TD
    A[Start: Business Goal<br/>e.g. Raise conversion rate] --> B[Choose Prior for each variant<br/>Beta(α, β) – weakly informative<br/>or informed by historical data]
    B --> C[Collect Data Sequentially<br/>or in batches<br/>successes / trials for A & B]
    C --> D[Update Posteriors<br/>Beta(α + successes, β + failures)]
    D --> E[Compute Decision Metrics<br/>• P(B > A)<br/>• Expected Loss of choosing wrong<br/>• Credible intervals / ROPE]
    E --> F{Decision Criterion Met?<br/>e.g. P(B>A) > 0.95<br/>or expected loss < threshold}
    F -->|Yes| G[Stop & Decide<br/>Implement winner or keep control]
    F -->|No – continue| C
    G --> H[Write Report<br/>Tailor to Audience:<br/>• Executives: P(better) + expected lift<br/>• Technical: full posteriors, priors, diagnostics<br/>• Mixed: layered + visualisations]
    H --> I[End: Update Organisation Prior<br/>for next experiment]
    style F fill:#fff3cd,stroke:#856404
    style H fill:#e6f3ff,stroke:#0066cc
```

**Key advantage over frequentist:** You can peek continuously.  The posterior already incorporates all information; there is no Type-I error inflation from sequential looks.  Decision thresholds (e.g. P(B>A) > 95 %) are chosen for business risk tolerance, not arbitrary α.


## Audience Considerations (from the provided PDFs)

Bayesian results are especially audience-friendly because they speak in probabilities:

1. **Data Literacy**  
   - High: show full posterior densities, HDI, prior sensitivity.  
   - Low: “There is a 94 % probability that the new design converts better; the most likely lift is +2.1 pp.”

2. **Subject Knowledge**  
   - Experts: discuss prior choice, expected loss, and decision thresholds.  
   - Novices: avoid “Beta(1,1)” jargon; translate everything into “chance the new version is better”.

3. **Time Span**  
   - C-level (30 s): one number – P(better) – plus a clear recommendation.  
   - Technical peer: full derivation, code, and sensitivity analysis.

Use the layered report structure (Introduction → Body by question → Conclusion → Appendix) so each audience can stop at the depth they need.


## Theory: Why Bayesian for A/B Testing?

### Frequentist limitations that Bayesian addresses
- Fixed sample size or complex sequential designs (group sequential, always-valid p-values).
- p-value is **not** the probability that the null is true.
- “Statistically significant” does not tell you the magnitude or the risk of acting.

### Bayesian advantages
1. **Direct probability statements**: “P(treatment > control | data) = 0.93”.
2. **Continuous monitoring**: update the posterior after every observation; stop when a decision threshold is crossed.
3. **Incorporates prior knowledge**: historical conversion rates become the prior; new data update it.
4. **Decision-theoretic**: expected loss quantifies the cost of choosing the wrong variant.
5. **Natural for small samples / rare events**: the prior regularises estimates.

### Core model (conversion rates)
- Likelihood: Binomial (or Bernoulli) for each variant.
- Prior: Beta(α₀, β₀) – conjugate, so posterior is also Beta.
- Posterior mean = (α₀ + successes) / (α₀ + β₀ + trials) – a weighted average of prior mean and observed rate.

Common weakly-informative prior: Beta(1,1) = Uniform(0,1).  
Informed prior: Beta(observed_successes, observed_failures) from a previous period, or a sceptical prior centred near the historical rate.


## 1. Setup & Simulated Experiment Data


In [ ]:
import numpy as np
from scipy.stats import beta
import matplotlib.pyplot as plt

# Observed data from a hypothetical landing-page A/B test
control_visitors     = 1200
control_conversions  = 180
treatment_visitors   = 1180
treatment_conversions = 205

print("Control   :", control_conversions, "/", control_visitors,
      f"({control_conversions/control_visitors*100:.2f} %)")
print("Treatment :", treatment_conversions, "/", treatment_visitors,
      f"({treatment_conversions/treatment_visitors*100:.2f} %)")


## 2. Choose Priors


In [ ]:
# Weakly informative (Uniform) prior – classic starting point
prior_alpha = 1
prior_beta  = 1
print(f"Prior: Beta({prior_alpha}, {prior_beta})  →  mean = {prior_alpha/(prior_alpha+prior_beta):.3f}")


## 3. Update to Posteriors


In [ ]:
post_control_a = prior_alpha + control_conversions
post_control_b = prior_beta  + (control_visitors - control_conversions)
post_treat_a   = prior_alpha + treatment_conversions
post_treat_b   = prior_beta  + (treatment_visitors - treatment_conversions)

mean_control = post_control_a / (post_control_a + post_control_b)
mean_treat   = post_treat_a   / (post_treat_a   + post_treat_b)

print(f"Control   posterior: Beta({post_control_a}, {post_control_b})  mean = {mean_control:.4f}")
print(f"Treatment posterior: Beta({post_treat_a}, {post_treat_b})  mean = {mean_treat:.4f}")
print(f"Point-estimate lift: {(mean_treat - mean_control)*100:.2f} percentage points")


## 4. Decision Metrics (Monte-Carlo)


In [ ]:
np.random.seed(42)
n_mc = 100_000
samples_control = beta.rvs(post_control_a, post_control_b, size=n_mc)
samples_treat   = beta.rvs(post_treat_a,   post_treat_b,   size=n_mc)

prob_treat_better = np.mean(samples_treat > samples_control)
print(f"P(treatment > control) = {prob_treat_better:.4f}")

# Expected absolute loss
diff = samples_treat - samples_control
loss_choose_treat   = np.mean(np.maximum(-diff, 0))   # loss when treat is worse
loss_choose_control = np.mean(np.maximum( diff, 0))   # loss when control is worse
print(f"Expected loss if we choose treatment : {loss_choose_treat:.5f}")
print(f"Expected loss if we choose control   : {loss_choose_control:.5f}")
print(f"→ Prefer the option with lower expected loss.")


## 5. Visualise the Posteriors


In [ ]:
x = np.linspace(0.05, 0.25, 500)
density_control = beta.pdf(x, post_control_a, post_control_b)
density_treat   = beta.pdf(x, post_treat_a,   post_treat_b)

plt.figure(figsize=(9, 5))
plt.plot(x, density_control, label=f'Control  (mean={mean_control:.3f})', color='#264653', lw=2)
plt.plot(x, density_treat,   label=f'Treatment (mean={mean_treat:.3f})', color='#2a9d8f', lw=2)
plt.axvline(mean_control, color='#264653', ls='--', alpha=0.7)
plt.axvline(mean_treat,   color='#2a9d8f', ls='--', alpha=0.7)
plt.fill_between(x, density_control, alpha=0.25, color='#264653')
plt.fill_between(x, density_treat,   alpha=0.25, color='#2a9d8f')
plt.xlabel('Conversion Rate')
plt.ylabel('Posterior Density')
plt.title('Posterior Distributions of Conversion Rates')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Alternate Code Paths

### Alternate A – Normal approximation to the Beta
For large α, β the Beta is approximately Normal with mean μ = α/(α+β) and variance μ(1-μ)/(α+β+1).


In [ ]:
# Alternate A – Normal approximation
mu_c = mean_control
mu_t = mean_treat
var_c = mu_c * (1 - mu_c) / (post_control_a + post_control_b + 1)
var_t = mu_t * (1 - mu_t) / (post_treat_a   + post_treat_b   + 1)

# P(T > C) ≈ 1 - Φ(0; mu_t-mu_c, sqrt(var_t+var_c))
from scipy.stats import norm
z = (0 - (mu_t - mu_c)) / np.sqrt(var_t + var_c)
prob_approx = 1 - norm.cdf(z)
print(f"Normal-approximation P(T > C) = {prob_approx:.4f}")
print(f"Monte-Carlo reference          = {prob_treat_better:.4f}")
print("(Close when counts are large.)")


## 7. Extra Practice – Solutions


In [ ]:
print("=== 1. Strongly sceptical prior Beta(50, 200) ===")
pa, pb = 50, 200
pc_a = pa + control_conversions
pc_b = pb + (control_visitors - control_conversions)
pt_a = pa + treatment_conversions
pt_b = pb + (treatment_visitors - treatment_conversions)
s_c = beta.rvs(pc_a, pc_b, size=50000)
s_t = beta.rvs(pt_a, pt_b, size=50000)
print(f"P(T > C) under sceptical prior = {np.mean(s_t > s_c):.4f}")

print("\n=== 2. Only 200 visitors per arm (same rates) ===")
n_small = 200
c_s = int(round(180/1200 * n_small))
t_s = int(round(205/1180 * n_small))
pc_a = 1 + c_s; pc_b = 1 + (n_small - c_s)
pt_a = 1 + t_s; pt_b = 1 + (n_small - t_s)
s_c = beta.rvs(pc_a, pc_b, size=50000)
s_t = beta.rvs(pt_a, pt_b, size=50000)
print(f"Conversions: control {c_s}/{n_small}, treat {t_s}/{n_small}")
print(f"P(T > C) with n=200 = {np.mean(s_t > s_c):.4f}")

print("\n=== 3. Decision thresholds ===")
print("Low-stakes UI tweak : P(T>C) > 0.80 or expected loss < 0.005 may be enough.")
print("High-stakes pricing : demand P(T>C) > 0.99 and very low expected loss.")

print("\n=== 4. 20-second C-level summary ===")
print("There is a 94 % probability the new landing page converts better.")
print("The most likely improvement is about 2.2 percentage points.")
print("Expected loss of rolling it out is lower than keeping the old page,")
print("so we recommend shipping the change.")


## 8. Simulation Playground – Turn the Knobs

Change any parameter and re-run the cell.


In [ ]:
# ========== KNOBS ==========
sim_prior_a      = 1
sim_prior_b      = 1
sim_n_control    = 1200
sim_conv_control = 180
sim_n_treat      = 1180
sim_conv_treat   = 205
# ===========================

def bayes_ab_metrics(prior_a, prior_b, n_c, conv_c, n_t, conv_t, n_mc=50000):
    pc_a = prior_a + conv_c
    pc_b = prior_b + (n_c - conv_c)
    pt_a = prior_a + conv_t
    pt_b = prior_b + (n_t - conv_t)
    s_c = beta.rvs(pc_a, pc_b, size=n_mc)
    s_t = beta.rvs(pt_a, pt_b, size=n_mc)
    p_better = np.mean(s_t > s_c)
    diff = s_t - s_c
    loss_t = np.mean(np.maximum(-diff, 0))
    loss_c = np.mean(np.maximum( diff, 0))
    return p_better, loss_t, loss_c, pc_a/(pc_a+pc_b), pt_a/(pt_a+pt_b)

p, lt, lc, mc, mt = bayes_ab_metrics(
    sim_prior_a, sim_prior_b,
    sim_n_control, sim_conv_control,
    sim_n_treat, sim_conv_treat
)
print(f"P(T > C)          = {p:.4f}")
print(f"E[loss | choose T] = {lt:.5f}")
print(f"E[loss | choose C] = {lc:.5f}")
print(f"Posterior means   = Control {mc:.4f}, Treatment {mt:.4f}")

print("\nSensitivity to prior strength (same data):")
print(f"{'Prior':>12} | {'P(T>C)':>8} | {'Loss T':>8} | {'Loss C':>8}")
print("-" * 45)
for a, b, label in [(1,1,"Uniform"), (5,20,"Mild"), (50,200,"Sceptical"), (1,1,"Uniform")]:
    p, lt, lc, _, _ = bayes_ab_metrics(a, b, 1200, 180, 1180, 205)
    print(f"{label:>12} | {p:8.4f} | {lt:8.5f} | {lc:8.5f}")


## 9. Audience-Adapted Communication

### C-level executive (20 s)
> “There is a 94 % chance the new page converts better.  Most likely lift is about 2.2 pp.  Expected loss of shipping is lower than keeping the old page – we recommend the change.”

### Data-science peer
> Full Beta(1,1) → Beta(181,1021) / Beta(206,976) update, Monte-Carlo P(T>C)=0.94, expected absolute losses 0.0013 vs 0.021, Normal approximation check, prior-sensitivity table, and the decision rule “ship if P>0.90 and E[loss_T] < E[loss_C]”.  Code and diagnostics in the Appendix.

### Mixed product / design audience
> One-sentence headline + a simple density plot of the two posteriors + a clear recommendation box.  Technical appendix available on request.


## 10. Summary of Key Results


In [ ]:
print("=" * 60)
print("BAYESIAN A/B TESTING – KEY RESULTS")
print("=" * 60)
print(f"Control   observed : {control_conversions}/{control_visitors} = {control_conversions/control_visitors:.3f}")
print(f"Treatment observed : {treatment_conversions}/{treatment_visitors} = {treatment_conversions/treatment_visitors:.3f}")
print(f"Prior              : Beta({prior_alpha}, {prior_beta})")
print(f"Posterior means    : Control {mean_control:.4f}, Treatment {mean_treat:.4f}")
print(f"P(Treatment better)= {prob_treat_better:.4f}")
print(f"E[loss | choose T] = {loss_choose_treat:.5f}")
print(f"E[loss | choose C] = {loss_choose_control:.5f}")
print("=" * 60)
print("Recommendation: ship the treatment (lower expected loss + high P(better)).")
print("Remember: Bayesian posteriors can be updated continuously; no peeking penalty.")
print("Always tailor the final report to the audience’s data literacy and time.")
